In [1]:
import torch

In [4]:
energy_func_gmm2 = lambda x: (0.8 * torch.exp(- torch.linalg.norm(x - 3., axis=1) ** 2 / 2)
                  + 0.2 * torch.exp(- torch.linalg.norm(x + 3., axis=1) ** 2 /2 ))   

In [ ]:
def log_expectation_reward(
        self,
        t: torch.Tensor,
        x: torch.Tensor,
        energy_function,
        num_mc_samples: int,
):
    repeated_x = x.unsqueeze(0).repeat_interleave(num_mc_samples, dim=0)

    samples = self.reverse_sample(repeated_x, t)

    log_rewards = energy_function(samples)

    return torch.logsumexp(log_rewards, dim=-1) - np.log(num_mc_samples)

def estimate_grad_Rt(
        self,
        t: torch.Tensor,
        x: torch.Tensor,
        energy_function,
        num_mc_samples: int = 20,
):
    if t.ndim == 0:
        t = t.unsqueeze(0).repeat(len(x))

    grad_fxn = torch.func.grad(self.log_expectation_reward, argnums=1)
    vmapped_fxn = torch.vmap(grad_fxn, in_dims=(0, 0, None, None), randomness="different")

    return vmapped_fxn(t, x, energy_function, num_mc_samples)

In [23]:
x = torch.tensor([[1.3, 1.3]], requires_grad=True)
true_score = torch.autograd.grad(energy_func_gmm2(x), x)[0]
true_score

tensor([[0.0756, 0.0756]])

In [30]:
size = 500
t = 0.01
samples = x - torch.randn([size, 2]) * t
samples.requires_grad_(True)
log_rewards = energy_func_gmm2(samples)
log_rewards.shape
lse = torch.logsumexp(log_rewards, dim=-1)
torch.autograd.grad(lse, x)

(tensor([[0.0756, 0.0755]]),)